In [ ]:
# editable install
# run `pip install -e .` in the project root

In [ ]:
import torch
import numpy as np
from scipy.io import loadmat

from torchbci.algorithms.kilosort import *
from torchbci.block.functional import get_channel_relative_distances

data_dir = '../data/'

input_dir = data_dir + 'c46_npx_raw.bin'
ground_truth_spikes_dir = data_dir + 'c46_all_spikes.mat'

c46_sampling_rate = 50023.87552924
ground_truth_spikes_sampling_rate = 50000

In [ ]:
def random_data_selection(numpy_data: np.ndarray, 
                          num_channels: int, 
                          num_samples: int, 
                          to_torch: bool = False, 
                          delete_numpy: bool = False):
    if num_channels == numpy_data.shape[0]: # all channels
        start_channel = 0 
    else:
        start_channel = np.random.randint(0, numpy_data.shape[0] - num_channels)
    if num_samples == numpy_data.shape[1]: # all samples
        start_index = 0
    else:
        start_index = np.random.randint(0, numpy_data.shape[1] - num_samples)
    selected_data = numpy_data[start_channel:start_channel + num_channels, start_index:start_index + num_samples].copy()
    if to_torch:
        selected_data = torch.from_numpy(selected_data).clone().float()
    if delete_numpy:
        del numpy_data
    return selected_data, start_index, start_index + num_samples, start_channel, start_channel + num_channels

n_rows = 384
data_memmap = np.memmap(input_dir, dtype=np.int16, mode='r', order='C')
data_array = data_memmap.reshape((n_rows, -1), order='F') #

print(f"Total raw data shape: {data_array.shape}")



In [ ]:
# Ground-truth can come from mat or npy
if ground_truth_spikes_dir.endswith('.mat'):
    # Load .mat file
    ground_truth_spikes_data = loadmat(ground_truth_spikes_dir)
    ground_truth_spikes = ground_truth_spikes_data['spk_times'].flatten()
elif ground_truth_spikes_dir.endswith('.npy'):
    ground_truth_spikes = np.load(ground_truth_spikes_dir, allow_pickle=True)
else:
    raise ValueError(f"Unsupported file format for ground truth spikes: {ground_truth_spikes_dir}")
    
# convert ground truth sampling rate to cell sampling rate as they are different
ground_truth_spikes = (ground_truth_spikes * (ground_truth_spikes_sampling_rate  / c46_sampling_rate)).astype(int)

last_index = ground_truth_spikes.max() + 100
print(f"last_index: {last_index}")
data_array = data_array[:, :last_index]
print(f"data_array.shape: {data_array.shape}")



In [ ]:
num_channels = 384
num_samples = 20000
data_tensor, start_index, end_index, start_channel, end_channel = random_data_selection(data_array, 
                                                                                        num_channels, 
                                                                                        num_samples, 
                                                                                        to_torch=True, 
                                                                                        delete_numpy=True)

# select spikes that based on the selected slice of raw data
selected_spikes = (ground_truth_spikes >= start_index) & (ground_truth_spikes < end_index)
ground_truth_spikes_raw = ground_truth_spikes.copy()
ground_truth_spikes = ground_truth_spikes[selected_spikes]
spikes_times_relative = ground_truth_spikes - start_index
print(spikes_times_relative)

In [ ]:
# templates = torch.tensor([[1, 1, 1, 1, 64, 1, 1, 1, 1],
#                           [1, 1, 1, 32, 64, 32, 1, 1, 1],
#                           [1, 1, 21, 42, 64, 42, 21, 1, 1],
#                           [1, 16, 32, 48, 64, 48, 32, 16, 1],
#                           [16, 26, 39, 52, 64, 52, 39, 26, 16],
#                           [16, 26, 42, 58, 64, 58, 42, 26, 16]], dtype=torch.float32)

templates_dir = '../data/wTEMP.npz'
templates = np.load(templates_dir)['wTEMP']
templates = torch.from_numpy(templates).float()

# templates_scaling = 10
# templates = templates * templates_scaling


In [ ]:
# get relative distances between channels from the channel mapping file
def generate_channel_coords(num_repeats):
    x_vals = np.tile([33, 0, 45, 12], num_repeats)
    y_vals = np.repeat(np.arange(20, 2 * (num_repeats * 20) + 20, 20), 2)
    return np.column_stack((x_vals, y_vals))

channel_coords = torch.tensor(generate_channel_coords(num_channels // 4), dtype=torch.float32)
channel_relative_distances = get_channel_relative_distances(channel_coords)

In [ ]:
kilosort4_sort_pipe = Kilosort4Algorithm(
    sample_rate=c46_sampling_rate,
    channel_relative_distances=channel_relative_distances,
    high_pass_filter_cutoff=300,
    detection_threshold=5,
    lookbehind_length=16,
    lookahead_length=64,
    num_feature_channels=16,
    max_spikes=128,
    feature_length=128,
    n_clusters=16,
    predefined_templates=templates,
    spike_template_matching_threshold_similarity=0.8,
    dim_pc_features=2,
)

In [ ]:
# x, spikes, spike_features, spike_pc_features = kilosort4_sort_pipe.forward(data_tensor)
clusters, centroid, cluster_meta = kilosort4_sort_pipe.forward(data_tensor)

In [ ]:
centroids_picked = []
clusters_meta_picked = []
clusters_picked = []

n_clustered_spikes = sum([len(meta) for meta in cluster_meta])
print(f"Number of detected spikes: {n_clustered_spikes}")
minimum_spikes_cluster = 3
for cluster, centroid, meta in zip(clusters, centroid, cluster_meta):
    if len(meta) > minimum_spikes_cluster:
        meta = sorted(meta, key=lambda x: x[1])
        print(meta)
        centroids_picked.append(centroid)
        clusters_meta_picked.append(meta)
        clusters_picked.append(cluster)
print(spikes_times_relative)

In [ ]:
# PCA on the clusters for visualization, mark the centroids
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
pca = PCA(n_components=2)
all_clustered_spikes = [torch.stack(c).numpy() for c in clusters_picked]
all_clustered_spikes_pca = [pca.fit_transform(np.concatenate(all_clustered_spikes))]

# fit once on all data
pca.fit(np.concatenate(all_clustered_spikes))
centroids_pca = pca.transform(torch.stack(centroids_picked).numpy())

plt.figure(figsize=(6, 5))
colors = plt.cm.tab20(np.linspace(0, 1, len(clusters_picked)))  # distinct colors

start = 0
for i, cluster in enumerate(clusters_picked):
    n = len(cluster)
    cluster_pca = pca.transform(torch.stack(cluster).numpy())
    plt.scatter(cluster_pca[:, 0], cluster_pca[:, 1], s=1, color=colors[i], label=f'C{i}')
    start += n

plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], s=100, c='black', marker='x')
plt.title("Kilosort's Clusters")
plt.legend(markerscale=4)
plt.show()



In [ ]:
# Plot spikes from a specific cluster (overlap spikes)
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
for spike in clusters_picked[1]:
    plt.plot(spike.numpy(), color='gray', alpha=0.5)
plt.title('Spikes from Cluster')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.show()

In [ ]:
# plot each cluster (x axis: time, y axis: channel) with a unique color
import matplotlib.pyplot as plt

colors = plt.get_cmap('tab20', len(clusters_meta_picked))
plt.figure(figsize=(15, 10))
for cluster_idx, meta in enumerate(clusters_meta_picked):
    color = colors(cluster_idx)
    for m in meta:
        channel, time = m
        plt.scatter(time, channel, color=color, s=10)

# gt spikes as vertical lines
for spike in spikes_times_relative:
    plt.axvline(x=spike, color='k', linestyle='--', alpha=0.3)

plt.ylim(175, 185)